<a href="https://colab.research.google.com/github/ahmedyomiisiaka/RP-Network-Analysis/blob/main/notebooks/05_Context_Specific_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount("/content/drive")

DIRS = [
    "/content/rp_network/data/raw",
    "/content/rp_network/data/processed",
    "/content/rp_network/data/results",
    "/content/rp_network/data/network",
    "/content/rp_network/docs",
    "/content/rp_network/figures",
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

drive_path = Path("/content/drive/MyDrive/rp_network")
colab_path = Path("/content/rp_network")

for f in drive_path.rglob("*"):
    if f.is_file():
        dest = colab_path / f.relative_to(drive_path)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dest)

print("✓ All files restored from Google Drive.")
print()
for f in sorted(colab_path.rglob("*.tsv")):
    print(f"  {f.relative_to(colab_path)}")

Mounted at /content/drive
✓ All files restored from Google Drive.

  data/processed/ppi_interactions.tsv
  data/processed/ppi_nodes.tsv
  data/raw/compartments_human.tsv
  data/raw/string_checkpoint.tsv
  data/raw/string_interactions_raw.tsv


# Notebook 05 — Context-Specific Network Filtering

**Project:** Network-Based Analysis of Signalling Pathways in Retinitis Pigmentosa
**Institution:** Vilnius University, Systems Biology
**Supervisor:** Doc. Dr. Erinija Pranckevičienė
**Author:** Ahmed Yomi Isiaka
**Stage:** 5 — Context-Specific Network Filtering
**Date:** 2026-08-13

---

## 1. Objective

To filter the raw PPI network from Stage 4 to retain only
interactions that are biologically plausible in the retinal
photoreceptor context.

The Stage 4 network contains 9,835 interactions among 4,132
proteins retrieved at STRING confidence score ≥ 0.400. Not all
of these interactions are active in retinal tissue — many are
generic interactions documented in non-retinal cell types or
subcellular compartments that do not co-occur in photoreceptors.

This notebook applies two sequential biological filters:

**Filter 1 — Subcellular co-localisation (COMPARTMENTS)**
Both interacting proteins must be present in the same
subcellular compartment. Proteins in different compartments
cannot physically interact.

**Filter 2 — Retinal tissue expression (Human Protein Atlas)**
Both interacting proteins must be expressed in retinal tissue.
Proteins not expressed in the retina are not relevant to
retinal disease biology.

An interaction passes the filter only if it satisfies
BOTH criteria.

## Supervisor guidance (2026-08-13)

> "Please focus on what we were discussing — databases to filter
> interactions: COMPARTMENTS and other. The lowest threshold is
> good since we will have to retain interactions that are in the
> same compartment and tissue."

This confirms:
- Working network: score ≥ 0.400 (9,835 edges, 4,132 proteins)
- Filter 1: COMPARTMENTS database — subcellular co-localisation
- Filter 2: Human Protein Atlas — retinal tissue expression

---
## 2. Background and Rationale

### Why context-specific filtering is necessary

Protein-protein interaction databases such as STRING aggregate
interaction evidence from many cell types, tissues, and
experimental conditions. A high STRING score means an interaction
is well-supported by evidence — but that evidence may come from
liver cells, cancer cell lines, or yeast two-hybrid experiments
that do not reflect what happens in a retinal photoreceptor.

For this project, we are specifically interested in:
- Interactions that occur in retinal photoreceptor cells
- Proteins that are physically co-located (same compartment)
- Proteins that are actually expressed in retinal tissue

Filtering by these biological criteria produces a smaller,
higher-quality network that is specific to retinal RP biology
rather than generic human cell biology.

### The two filters

**Filter 1 — Subcellular co-localisation**
Physical protein-protein interaction requires that both proteins
are present in the same subcellular compartment at the same time.
For example, a membrane protein cannot directly interact with a
strictly nuclear protein. The COMPARTMENTS database provides
confidence-scored localisation data for human proteins across
all major subcellular compartments.

**Filter 2 — Retinal tissue expression**
Only proteins expressed in retinal tissue are relevant to RP
biology. The Human Protein Atlas provides RNA expression data
across 54 human tissues including the retina, scored as:
Not detected / Low / Medium / High / Enhanced.
We retain only proteins with detectable retinal expression.

### Filter order

Filter 1 (COMPARTMENTS) is applied first because:
- It is a direct biological constraint on physical interaction
- It does not depend on disease state — a protein in the nucleus
  cannot interact with a membrane protein regardless of tissue
- COMPARTMENTS data is downloadable as a flat file — efficient

Filter 2 (Human Protein Atlas) is applied to the output of
Filter 1 — a smaller network, making the query faster.

---
## 4. Methodological Notes — All Databases Used

---

### Database 1 — COMPARTMENTS (Jensen Lab)

**What it is:**
COMPARTMENTS is a weekly-updated database of protein subcellular
localisation evidence integrating experimental data, curated
databases (UniProt, Gene Ontology), text mining, and sequence-
based prediction. Each protein-compartment pair is assigned a
confidence score from 1 (weak) to 5 (strong).

**How it was queried:**
Downloaded as a flat TSV file from:
https://download.jensenlab.org/human_compartment_integrated_full.tsv
Access date: 2026-08-13
No API call required — direct file download.

**What information was retrieved:**
For each human protein: which subcellular compartments it
localises to, and the confidence score for each localisation.

**How it was used:**
For each interaction edge (protein A — protein B):
- Retrieved all compartments for protein A (score ≥ 3)
- Retrieved all compartments for protein B (score ≥ 3)
- If the two compartment sets overlap → edge RETAINED
- If no overlap exists → proteins cannot co-localise → edge REMOVED

**Score threshold:** ≥ 3 (medium-high confidence)

**Citation:**
Binder JX, et al. (2014). COMPARTMENTS: unification and
visualization of protein subcellular localization evidence.
Database, bau012. https://doi.org/10.1093/database/bau012

---

### Database 2 — Human Protein Atlas (HPA)

**What it is:**
The Human Protein Atlas provides RNA and protein expression data
across 54 human tissues including the retina, combining RNA
sequencing (nTPM values) and antibody-based protein profiling.

**How it was queried:**
Downloaded as a flat TSV file from:
https://www.proteinatlas.org/about/download
File: rna_tissue_consensus.tsv
Access date: 2026-08-13
Contains nTPM values for all human genes across all tissues.

**What information was retrieved:**
For each protein: nTPM (normalised transcripts per million)
expression value specifically in retinal tissue.

**How it was used:**
For each protein in the network after COMPARTMENTS filtering:
- Retrieved nTPM value in retina
- nTPM > 0 → protein expressed in retina → RETAINED
- nTPM = 0 or not found → not expressed → edge FLAGGED
An edge is kept only if BOTH proteins have retinal expression.

**Citation:**
Uhlén M, et al. (2015). Tissue-based map of the human proteome.
Science, 347(6220):1260419.
https://doi.org/10.1126/science.1260419

---

### Database 3 — IID (Integrated Interactions Database)

**What it is:**
IID provides tissue-specific protein-protein interaction data for
human and model organisms. Interactions are assigned to tissues
where their protein pairs or encoding genes are expressed.
It covers 30 tissues in humans including retina.

**How it was queried:**
Queried via the IID web interface at:
https://iid.ophid.utoronto.ca
Input: protein list, Species: Homo sapiens, Tissue: Retina
Access date: 2026-08-13

**What information was retrieved:**
Which interactions in our network have documented evidence
of activity in retinal tissue specifically.

**How it was used:**
For each edge remaining after Filters A and B:
- Checked whether IID reports this interaction as retina-active
- IID-supported → edge annotated as "retina-confirmed"
- Not in IID → edge annotated as "retina-unconfirmed"
Both are retained but annotation allows downstream filtering.

**Citation:**
Kotlyar M, et al. (2019). IID 2018 update: context-specific
physical protein-protein interactions. Nucleic Acids Res,
47(D1):D581-D589. https://doi.org/10.1093/nar/gky1037

---

### Database 4 — TissueNet v3

**What it is:**
TissueNet provides tissue-resolved scores for human PPI networks
by integrating gene expression data from multiple sources to
assign each interaction a tissue-specific activity score.

**How it was queried:**
Downloaded from: https://netbio.bgu.ac.il/tissuenet
Access date: 2026-08-13
File contains per-tissue interaction scores for human PPIs.

**What information was retrieved:**
For each interaction: a retinal tissue-specific activity score.

**How it was used:**
Used as additional evidence to score/rank interactions by their
retinal tissue relevance. Higher TissueNet retinal score →
higher confidence the interaction is active in the retina.

**Citation:**
Basha O, et al. (2015). TissueNet: The database of human tissue
protein-protein interactions. Nucleic Acids Res, 43(D1):D424-D429.

---

### Comparison of Databases

| Database | Type | Retina-specific | Format | Used for |
|----------|------|----------------|--------|----------|
| COMPARTMENTS | Subcellular localisation | No (all compartments) | TSV download | Filter A |
| Human Protein Atlas | Tissue expression | Yes (retina nTPM) | TSV download | Filter B |
| IID | Tissue-specific PPI | Yes (retina PPIs) | Web query | Filter C |
| TissueNet v3 | Tissue-resolved PPI scores | Yes (retina score) | TSV download | Filter D |

**Why we use all four:**
Each database addresses a different biological question.
COMPARTMENTS tells us IF proteins can physically meet.
HPA tells us IF proteins are present in retinal tissue.
IID tells us IF this specific interaction occurs in retina.
TissueNet tells us HOW STRONGLY the interaction is supported
in retinal tissue. Together they provide a multi-evidence
filter that is much more reliable than any single database.

---
## 5. Methodological Decisions

| Item | Decision | Status |
|------|----------|--------|
| Input network | ppi_interactions.tsv (9,835 edges) | Confirmed |
| STRING threshold | 0.400 (supervisor confirmed 2026-08-13) | Confirmed |
| Filter A | COMPARTMENTS — subcellular co-localisation | Confirmed |
| Filter B | Human Protein Atlas — retinal nTPM expression | Confirmed |
| Filter C | IID — retina-specific interaction evidence | Confirmed |
| Filter D | TissueNet v3 — retinal interaction scoring | Confirmed |
| COMPARTMENTS score cutoff | ≥ 3 (medium-high confidence) | Proposed |
| HPA expression cutoff | nTPM > 0 (any detectable expression) | Proposed |
| Filter order | A → B → C → D | Confirmed |
| Edge retention rule | Must pass Filter A AND Filter B | Confirmed |
| IID/TissueNet role | Annotation and ranking, not hard removal | Proposed |
| Seed protein handling | Seed proteins retained regardless | To confirm |

---

## 6. Output Files

| File | Description |
|------|-------------|
| `data/raw/compartments_human.tsv` | Raw COMPARTMENTS download |
| `data/raw/hpa_rna_retina.tsv` | HPA retinal expression data |
| `data/processed/filter_A_compartments.tsv` | Edges passing compartment filter |
| `data/processed/filter_B_expression.tsv` | Edges passing expression filter |
| `data/processed/filter_C_iid.tsv` | IID retinal interaction annotations |
| `data/processed/filter_D_tissuenet.tsv` | TissueNet retinal scores |
| `data/processed/network_filtered.tsv` | Final filtered interaction table |
| `data/processed/network_filtered_nodes.tsv` | Final filtered node table |
| `data/network/rp_network_filtered.graphml` | Final filtered network |

In [2]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 1: Install libraries and create folders
# ─────────────────────────────────────────────────────────────────

import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "requests", "pandas", "networkx"],
    check=True
)

from pathlib import Path

DIRS = [
    "/content/rp_network/data/raw",
    "/content/rp_network/data/processed",
    "/content/rp_network/data/results",
    "/content/rp_network/data/network",
    "/content/rp_network/docs",
    "/content/rp_network/figures",
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✓ Libraries installed.")
print("✓ Folder structure ready.")

✓ Libraries installed.
✓ Folder structure ready.


In [3]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 2: Imports
# ─────────────────────────────────────────────────────────────────

import requests
import pandas as pd
import networkx as nx
import time
from pathlib import Path
from datetime import datetime

print(f"pandas   : {pd.__version__}")
print(f"networkx : {nx.__version__}")
print(f"requests : {requests.__version__}")
print("✓ All imports successful.")

pandas   : 2.2.2
networkx : 3.6.1
requests : 2.32.4
✓ All imports successful.


In [4]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 3: Load the Stage 4 interaction network
#
# Input : data/processed/ppi_interactions.tsv
#         9,835 unique edges, 4,132 proteins
#         STRING confidence score >= 0.400
# ─────────────────────────────────────────────────────────────────

INTERACTION_FILE = "/content/rp_network/data/processed/ppi_interactions.tsv"
NODE_FILE        = "/content/rp_network/data/processed/ppi_nodes.tsv"

df_interactions = pd.read_csv(INTERACTION_FILE, sep="\t")
df_nodes        = pd.read_csv(NODE_FILE, sep="\t")

# Rebuild seed gene set
seed_genes = df_nodes[df_nodes["node_type"] == "Seed"]["protein_name"].tolist()
seed_upper = set(g.upper() for g in seed_genes)

print("=" * 55)
print("STAGE 4 NETWORK — LOADED")
print("=" * 55)
print(f"Interaction edges : {len(df_interactions)}")
print(f"Total proteins    : {len(df_nodes)}")
print(f"  Seed proteins   : {len(seed_genes)}")
print(f"  New partners    : {len(df_nodes) - len(seed_genes)}")
print()
print("Score distribution (confirming threshold = 0.400):")
bins = [(0.4,0.5,"0.400-0.499"), (0.5,0.7,"0.500-0.699"),
        (0.7,0.9,"0.700-0.899"), (0.9,1.01,"0.900-1.000")]
for lo, hi, label in bins:
    n = ((df_interactions["score"] >= lo) &
         (df_interactions["score"] < hi)).sum()
    print(f"  {label} : {n} edges")
print()
print("✓ Network loaded and ready for filtering.")

STAGE 4 NETWORK — LOADED
Interaction edges : 9835
Total proteins    : 4132
  Seed proteins   : 105
  New partners    : 4027

Score distribution (confirming threshold = 0.400):
  0.400-0.499 : 3590 edges
  0.500-0.699 : 3410 edges
  0.700-0.899 : 1635 edges
  0.900-1.000 : 1200 edges

✓ Network loaded and ready for filtering.


In [5]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 4: Download COMPARTMENTS database
#
# Database : COMPARTMENTS (Jensen Lab, University of Copenhagen)
# File     : human_compartment_integrated_full.tsv
# URL      : https://download.jensenlab.org/
# Access   : 2026-08-13
# Size     : ~200MB — download takes 1-3 minutes in Colab
#
# File format (tab-separated):
#   Col 1: Protein identifier (Ensembl gene ID or gene name)
#   Col 2: Protein name (gene symbol)
#   Col 3: GO term ID (compartment)
#   Col 4: GO term name (compartment name)
#   Col 5: Evidence score (1-5, higher = more confident)
#   Col 6: Evidence source (knowledge, experiments, textmining, predictions)
#
# Score interpretation:
#   5 = strongest evidence (manually curated, multiple sources)
#   4 = strong evidence
#   3 = medium-high evidence ← our cutoff
#   2 = medium evidence
#   1 = weak evidence (single text mining mention)
#
# We use score >= 3 to retain well-supported localisations
# while excluding weak or single-source predictions.
# ─────────────────────────────────────────────────────────────────

COMPARTMENTS_URL  = "https://download.jensenlab.org/human_compartment_integrated_full.tsv"
COMPARTMENTS_FILE = "/content/rp_network/data/raw/compartments_human.tsv"
COMPARTMENTS_SCORE_CUTOFF = 3

print("Downloading COMPARTMENTS database...")
print(f"URL  : {COMPARTMENTS_URL}")
print(f"File : {COMPARTMENTS_FILE}")
print("This may take 1-3 minutes...")
print()

r = requests.get(COMPARTMENTS_URL, timeout=300, stream=True)
print(f"Status : {r.status_code}")

if r.status_code == 200:
    with open(COMPARTMENTS_FILE, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    size_mb = Path(COMPARTMENTS_FILE).stat().st_size / 1_048_576
    print(f"✓ Downloaded: {size_mb:.1f} MB")
else:
    print(f"✗ Error: {r.status_code} — {r.text[:200]}")

URL  : https://download.jensenlab.org/human_compartment_integrated_full.tsv
File : /content/rp_network/data/raw/compartments_human.tsv
This may take 1-3 minutes...

Status : 200
✓ Downloaded: 288.4 MB


In [6]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 5: Parse COMPARTMENTS database
#
# Load the COMPARTMENTS file and filter for:
#   - Human proteins only (already human-specific file)
#   - Score >= COMPARTMENTS_SCORE_CUTOFF (>= 3)
#
# Build a lookup dictionary:
#   protein_name → set of compartment GO terms
#
# This dictionary will be used to check compartment overlap
# for each interaction edge.
# ─────────────────────────────────────────────────────────────────

print("Loading COMPARTMENTS database...")
print(f"Score cutoff: >= {COMPARTMENTS_SCORE_CUTOFF}")
print()

# Load — no header in this file
df_comp = pd.read_csv(
    COMPARTMENTS_FILE,
    sep="\t",
    header=None,
    names=["ensembl_id", "gene_symbol", "go_term_id",
           "go_term_name", "score", "evidence_source"],
    dtype={"score": float},
    low_memory=False
)

print(f"Total rows in COMPARTMENTS file : {len(df_comp):,}")
print(f"Unique proteins                 : {df_comp['gene_symbol'].nunique():,}")
print(f"Unique compartments             : {df_comp['go_term_name'].nunique():,}")
print()

# Filter by score
df_comp_filtered = df_comp[
    df_comp["score"] >= COMPARTMENTS_SCORE_CUTOFF
].copy()

print(f"After score >= {COMPARTMENTS_SCORE_CUTOFF} filter:")
print(f"  Rows remaining    : {len(df_comp_filtered):,}")
print(f"  Unique proteins   : {df_comp_filtered['gene_symbol'].nunique():,}")
print()

# Build lookup: gene_symbol (upper) → set of GO compartment terms
compartment_lookup = {}

for _, row in df_comp_filtered.iterrows():
    gene = str(row["gene_symbol"]).upper()
    comp = str(row["go_term_name"])
    if gene not in compartment_lookup:
        compartment_lookup[gene] = set()
    compartment_lookup[gene].add(comp)

print(f"Compartment lookup built for : {len(compartment_lookup):,} proteins")
print()

# Check a few well-known RP proteins
test_proteins = ["RHO", "RPGR", "PRPF8", "CRX", "GNAT1"]
print("Sample compartments for known RP proteins:")
for p in test_proteins:
    comps = compartment_lookup.get(p, set())
    print(f"  {p}: {sorted(comps)[:5]}")

Loading COMPARTMENTS database...
Score cutoff: >= 3

Total rows in COMPARTMENTS file : 4,859,940
Unique proteins                 : 28,545
Unique compartments             : 3,312

After score >= 3 filter:
  Rows remaining    : 388,097
  Unique proteins   : 19,342

Compartment lookup built for : 19,342 proteins

Sample compartments for known RP proteins:
  RHO: ['9+0 non-motile cilium', 'Anchoring junction', 'Bounding membrane of organelle', 'Cell junction', 'Cell periphery']
  RPGR: ['9+0 non-motile cilium', '9+2 motile cilium', 'Axoneme', 'Cell projection', 'Cellular anatomical entity']
  PRPF8: ['Catalytic step 1 spliceosome', 'Catalytic step 2 spliceosome', 'Cellular anatomical entity', 'Cytoplasm', 'Intracellular anatomical structure']
  CRX: ['Cellular anatomical entity', 'Chromatin', 'Chromosome', 'GO:0005623', 'GO:0044424']
  GNAT1: ['9+0 non-motile cilium', 'Bounding membrane of organelle', 'Cell periphery', 'Cell projection', 'Cell projection membrane']


In [8]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 6 (corrected): Apply Filter A — COMPARTMENTS
#
# Problem identified:
#   Broad GO parent terms (e.g. "cellular_component",
#   "Cellular anatomical entity") are assigned to almost all
#   proteins, making the overlap check trivially true for every pair.
#   This means the filter was not discriminating.
#
# Solution:
#   Exclude overly broad/generic GO compartment terms and use
#   only specific subcellular compartments that are biologically
#   meaningful for physical co-localisation.
#
# Specific compartments retained:
#   Nucleus, Cytoplasm, Mitochondrion, Endoplasmic reticulum,
#   Golgi apparatus, Plasma membrane, Lysosome, Peroxisome,
#   Cytosol, Cell junction, Cilium, Photoreceptor outer segment,
#   Extracellular space, Nucleolus, Centrosome, Flagellum,
#   Axoneme, Synaptic vesicle, Spliceosome, etc.
#
# Broad terms EXCLUDED:
#   cellular_component, Cellular anatomical entity,
#   Intracellular anatomical structure, membrane-bounded organelle,
#   Organelle, Intracellular organelle, etc.
# ─────────────────────────────────────────────────────────────────

# Define broad GO terms to EXCLUDE from compartment matching
# These are parent/root terms that apply to nearly all proteins
BROAD_TERMS_TO_EXCLUDE = {
    "cellular_component",
    "cellular anatomical entity",
    "organelle",
    "intracellular anatomical structure",
    "membrane-bounded organelle",
    "intracellular organelle",
    "intracellular membrane-bounded organelle",
    "cell",
    "protein-containing complex",
    "cellular_component",
    "go:0005623",   # cell (GO ID form)
    "go:0044424",   # intracellular part
    "go:0005622",   # intracellular
    "go:0043226",   # organelle
    "go:0043229",   # intracellular organelle
    "go:0043227",   # membrane-bounded organelle
    "go:0043231",   # intracellular membrane-bounded organelle
    "go:0044464",   # cell part
}

# Rebuild compartment lookup using only specific terms
print("Rebuilding compartment lookup with specific terms only...")

compartment_lookup_specific = {}

for _, row in df_comp_filtered.iterrows():
    gene      = str(row["gene_symbol"]).upper()
    comp_name = str(row["go_term_name"]).lower()
    comp_id   = str(row["go_term_id"]).lower()
    comp_orig = str(row["go_term_name"])   # original case for display

    # Skip broad terms
    if comp_name in BROAD_TERMS_TO_EXCLUDE:
        continue
    if comp_id in BROAD_TERMS_TO_EXCLUDE:
        continue

    if gene not in compartment_lookup_specific:
        compartment_lookup_specific[gene] = set()
    compartment_lookup_specific[gene].add(comp_orig)

print(f"Proteins with specific compartment data : {len(compartment_lookup_specific):,}")
print()

# Verify with known RP proteins
print("Sample specific compartments for known RP proteins:")
test_proteins = ["RHO", "RPGR", "PRPF8", "CRX", "GNAT1"]
for p in test_proteins:
    comps = sorted(compartment_lookup_specific.get(p, set()))
    print(f"  {p}: {comps[:6]}")
print()

# Apply filter with specific compartments
print("Applying Filter A with specific compartments...")

results_A = []

for _, row in df_interactions.iterrows():
    pA = str(row["preferredName_A"]).upper()
    pB = str(row["preferredName_B"]).upper()

    comps_A = compartment_lookup_specific.get(pA, set())
    comps_B = compartment_lookup_specific.get(pB, set())

    if not comps_A and not comps_B:
        status = "both_unknown"
        overlap = set()
    elif not comps_A:
        status = "protein_A_unknown"
        overlap = set()
    elif not comps_B:
        status = "protein_B_unknown"
        overlap = set()
    else:
        overlap = comps_A & comps_B
        status = "co_localised" if overlap else "no_overlap"

    results_A.append({
        "preferredName_A"      : row["preferredName_A"],
        "preferredName_B"      : row["preferredName_B"],
        "score"                : row["score"],
        "protein_A_is_seed"    : row["protein_A_is_seed"],
        "protein_B_is_seed"    : row["protein_B_is_seed"],
        "compartment_status"   : status,
        "shared_compartments"  : "; ".join(sorted(overlap)) if overlap else "",
        "n_shared_compartments": len(overlap),
    })

df_filter_A = pd.DataFrame(results_A)

print("=" * 55)
print("FILTER A — COMPARTMENTS RESULTS (specific terms)")
print("=" * 55)
print(f"Total edges input    : {len(df_filter_A):,}")
print()
print("Compartment status breakdown:")
print(df_filter_A["compartment_status"].value_counts().to_string())
print()

# Edges that PASS
df_pass_A = df_filter_A[
    df_filter_A["compartment_status"] != "no_overlap"
].copy()
df_fail_A = df_filter_A[
    df_filter_A["compartment_status"] == "no_overlap"
].copy()

print(f"Edges PASSING Filter A : {len(df_pass_A):,}")
print(f"  co_localised         : {len(df_filter_A[df_filter_A['compartment_status']=='co_localised']):,}")
print(f"  unknown (kept)       : {len(df_filter_A[df_filter_A['compartment_status'].str.contains('unknown')]):,}")
print(f"Edges REMOVED          : {len(df_fail_A):,}")
print()

# Most common specific shared compartments
from collections import Counter
all_comps = []
for comps in df_filter_A["shared_compartments"]:
    if comps:
        all_comps.extend(comps.split("; "))
comp_counts = Counter(all_comps)

print("Top 15 most common SPECIFIC shared compartments:")
for comp, count in comp_counts.most_common(15):
    print(f"  {comp:<45} : {count:>5} edges")

# Save
out_A = "/content/rp_network/data/processed/filter_A_compartments.tsv"
df_filter_A.to_csv(out_A, sep="\t", index=False)
df_pass_A.to_csv(
    "/content/rp_network/data/processed/filter_A_passed.tsv",
    sep="\t", index=False
)
print()
print(f"✓ Full filter A table → filter_A_compartments.tsv")
print(f"✓ Passing edges       → filter_A_passed.tsv")

Rebuilding compartment lookup with specific terms only...
Proteins with specific compartment data : 18,807

Sample specific compartments for known RP proteins:
  RHO: ['9+0 non-motile cilium', 'Anchoring junction', 'Bounding membrane of organelle', 'Cell junction', 'Cell periphery', 'Cell projection']
  RPGR: ['9+0 non-motile cilium', '9+2 motile cilium', 'Axoneme', 'Cell projection', 'Centrosome', 'Ciliary basal body']
  PRPF8: ['Catalytic step 1 spliceosome', 'Catalytic step 2 spliceosome', 'Cytoplasm', 'Intracellular organelle lumen', 'Membrane', 'Nuclear body']
  CRX: ['Chromatin', 'Chromosome', 'Intracellular non-membrane-bounded organelle', 'Nucleus', 'non-membrane-bounded organelle']
  GNAT1: ['9+0 non-motile cilium', 'Bounding membrane of organelle', 'Cell periphery', 'Cell projection', 'Cell projection membrane', 'Ciliary membrane']

Applying Filter A with specific compartments...
FILTER A — COMPARTMENTS RESULTS (specific terms)
Total edges input    : 9,835

Compartment status

In [10]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 7 (corrected): Download Human Protein Atlas
#
# Database : Human Protein Atlas v25.1
# File     : rna_tissue_consensus.tsv
# URL      : https://www.proteinatlas.org/download/
# Access   : 2026-08-13
#
# HPA v25.1 provides RNA consensus tissue expression data for
# 20,162 human genes across 51 tissues including retina.
# Expression values are in nTPM (normalised transcripts per million).
#
# Note: The file URL changed between HPA versions.
# Current correct URL confirmed from the HPA download page v25.1.
# ─────────────────────────────────────────────────────────────────

HPA_URL  = "https://www.proteinatlas.org/download/tsv/rna_tissue_consensus.tsv.zip"
HPA_FILE = "/content/rp_network/data/raw/hpa_rna_consensus.tsv.zip"

print("Downloading Human Protein Atlas RNA consensus data (v25.1)...")
print(f"URL  : {HPA_URL}")
print("This may take 1-3 minutes...")
print()

r_hpa = requests.get(HPA_URL, timeout=300, stream=True)
print(f"Status : {r_hpa.status_code}")

if r_hpa.status_code == 200:
    with open(HPA_FILE, "wb") as f:
        for chunk in r_hpa.iter_content(chunk_size=8192):
            f.write(chunk)
    size_mb = Path(HPA_FILE).stat().st_size / 1_048_576
    print(f"✓ Downloaded: {size_mb:.1f} MB")
    print(f"  Saved to: {HPA_FILE}")
elif r_hpa.status_code == 404:
    # Try alternative URL format
    print("First URL failed. Trying alternative...")
    HPA_URL2 = "https://www.proteinatlas.org/download/rna_tissue_consensus.tsv.zip"
    r_hpa2 = requests.get(HPA_URL2, timeout=300, stream=True)
    print(f"Status : {r_hpa2.status_code}")
    if r_hpa2.status_code == 200:
        with open(HPA_FILE, "wb") as f:
            for chunk in r_hpa2.iter_content(chunk_size=8192):
                f.write(chunk)
        size_mb = Path(HPA_FILE).stat().st_size / 1_048_576
        print(f"✓ Downloaded: {size_mb:.1f} MB")
    else:
        print("✗ Both URLs failed.")
        print()
        print("Manual download instructions:")
        print("1. Go to https://www.proteinatlas.org/about/download")
        print("2. Under 'Tissue' section, find 'RNA expression (consensus)'")
        print("3. Click the download link for the TSV file")
        print("4. Upload the downloaded zip file to Colab")
        print("5. Place it at: /content/rp_network/data/raw/hpa_rna_consensus.tsv.zip")
else:
    print(f"✗ Error: {r_hpa.status_code}")

URL  : https://www.proteinatlas.org/download/tsv/rna_tissue_consensus.tsv.zip
This may take 1-3 minutes...

Status : 200
✓ Downloaded: 5.0 MB
  Saved to: /content/rp_network/data/raw/hpa_rna_consensus.tsv.zip


In [11]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 8: Parse HPA and extract retinal expression
# ─────────────────────────────────────────────────────────────────

import zipfile

print("Extracting and parsing HPA data...")

# Extract zip
with zipfile.ZipFile(HPA_FILE, "r") as z:
    names = z.namelist()
    print(f"Files in zip: {names}")
    z.extractall("/content/rp_network/data/raw/")

# Find extracted file
extracted = [f for f in names if f.endswith(".tsv")][0]
extracted_path = f"/content/rp_network/data/raw/{extracted}"

print(f"Extracted: {extracted_path}")
print()

# Load full file
df_hpa = pd.read_csv(extracted_path, sep="\t", low_memory=False)
print(f"HPA file columns : {list(df_hpa.columns)}")
print(f"Total rows       : {len(df_hpa):,}")
print(f"Unique tissues   : {df_hpa.iloc[:,2].nunique()}")
print()

# Identify correct column names
tissue_col = df_hpa.columns[2]   # usually "Tissue"
gene_col   = df_hpa.columns[1]   # usually "Gene name"
ntpm_col   = df_hpa.columns[3]   # usually "nTPM"

# Show available tissue names containing "retina"
retina_tissues = df_hpa[
    df_hpa[tissue_col].str.lower().str.contains("retina", na=False)
][tissue_col].unique()

print(f"Retina-related tissue names found: {retina_tissues}")

Extracting and parsing HPA data...
Files in zip: ['rna_tissue_consensus.tsv']
Extracted: /content/rp_network/data/raw/rna_tissue_consensus.tsv

HPA file columns : ['Gene', 'Gene name', 'Tissue', 'nTPM']
Total rows       : 1,026,292
Unique tissues   : 51

Retina-related tissue names found: ['retina']


In [12]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 9: Apply Filter B — Retinal Expression (HPA)
#
# Filter rule:
#   Both proteins in an interaction must have nTPM > 0 in retina.
#   nTPM = 0 → not detected → protein not expressed in retina
#   nTPM > 0 → expressed in retina → retained
#
# Applied to the edges that PASSED Filter A.
# ─────────────────────────────────────────────────────────────────

# Extract retinal expression only
RETINA_TISSUE = retina_tissues[0] if len(retina_tissues) > 0 else "retina"

df_retina = df_hpa[
    df_hpa[tissue_col].str.lower().str.contains("retina", na=False)
][[gene_col, ntpm_col]].copy()

df_retina.columns = ["gene_symbol", "ntpm_retina"]
df_retina["gene_symbol"] = df_retina["gene_symbol"].str.upper()
df_retina = df_retina.drop_duplicates(subset=["gene_symbol"])

# Build lookup: gene → nTPM in retina
retina_expr = dict(zip(df_retina["gene_symbol"], df_retina["ntpm_retina"]))

print(f"Retinal expression data loaded:")
print(f"  Proteins with retinal data : {len(retina_expr):,}")
print(f"  Expressed (nTPM > 0)       : {sum(1 for v in retina_expr.values() if v > 0):,}")
print(f"  Not expressed (nTPM = 0)   : {sum(1 for v in retina_expr.values() if v == 0):,}")
print()

# Apply to edges passing Filter A
results_B = []

for _, row in df_pass_A.iterrows():
    pA = str(row["preferredName_A"]).upper()
    pB = str(row["preferredName_B"]).upper()

    ntpm_A = retina_expr.get(pA, None)
    ntpm_B = retina_expr.get(pB, None)

    # Determine expression status
    if ntpm_A is None and ntpm_B is None:
        expr_status = "both_not_in_HPA"
    elif ntpm_A is None:
        expr_status = "protein_A_not_in_HPA"
    elif ntpm_B is None:
        expr_status = "protein_B_not_in_HPA"
    elif ntpm_A > 0 and ntpm_B > 0:
        expr_status = "both_expressed"
    elif ntpm_A == 0 and ntpm_B == 0:
        expr_status = "both_not_expressed"
    elif ntpm_A == 0:
        expr_status = "protein_A_not_expressed"
    else:
        expr_status = "protein_B_not_expressed"

    results_B.append({
        **row.to_dict(),
        "ntpm_retina_A"    : ntpm_A,
        "ntpm_retina_B"    : ntpm_B,
        "expression_status": expr_status,
    })

df_filter_B = pd.DataFrame(results_B)

print("=" * 55)
print("FILTER B — RETINAL EXPRESSION RESULTS")
print("=" * 55)
print(f"Input edges (from Filter A) : {len(df_filter_B)}")
print()
print("Expression status breakdown:")
print(df_filter_B["expression_status"].value_counts().to_string())
print()

# Edges that PASS Filter B
df_pass_B = df_filter_B[
    df_filter_B["expression_status"].isin([
        "both_expressed",
        "both_not_in_HPA",        # unknown → keep conservatively
        "protein_A_not_in_HPA",   # unknown → keep conservatively
        "protein_B_not_in_HPA",   # unknown → keep conservatively
    ])
].copy()

df_fail_B = df_filter_B[
    df_filter_B["expression_status"].isin([
        "both_not_expressed",
        "protein_A_not_expressed",
        "protein_B_not_expressed",
    ])
].copy()

print(f"Edges PASSING Filter B     : {len(df_pass_B)}")
print(f"  Both expressed in retina : {len(df_filter_B[df_filter_B['expression_status']=='both_expressed'])}")
print(f"  Unknown (kept)           : {len(df_pass_B) - len(df_filter_B[df_filter_B['expression_status']=='both_expressed'])}")
print(f"Edges REMOVED              : {len(df_fail_B)}")
print()

# Save
out_B = "/content/rp_network/data/processed/filter_B_expression.tsv"
df_filter_B.to_csv(out_B, sep="\t", index=False)
df_pass_B.to_csv(
    "/content/rp_network/data/processed/filter_B_passed.tsv",
    sep="\t", index=False
)
print(f"✓ Full filter B table → filter_B_expression.tsv")
print(f"✓ Passing edges       → filter_B_passed.tsv")

Retinal expression data loaded:
  Proteins with retinal data : 20,151
  Expressed (nTPM > 0)       : 16,460
  Not expressed (nTPM = 0)   : 3,691

FILTER B — RETINAL EXPRESSION RESULTS
Input edges (from Filter A) : 9312

Expression status breakdown:
expression_status
both_expressed             8740
protein_B_not_expressed     389
protein_B_not_in_HPA        183

Edges PASSING Filter B     : 8923
  Both expressed in retina : 8740
  Unknown (kept)           : 183
Edges REMOVED              : 389

✓ Full filter B table → filter_B_expression.tsv
✓ Passing edges       → filter_B_passed.tsv


In [13]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 10: Filtering summary so far
#
# Summary of what Filters A and B have done to the network.
# ─────────────────────────────────────────────────────────────────

print("=" * 55)
print("FILTERING PROGRESS SUMMARY")
print("=" * 55)
print()
print(f"Original network (Stage 4)  : {len(df_interactions):,} edges")
print()
print(f"After Filter A (COMPARTMENTS):")
print(f"  Edges removed (no overlap): {len(df_fail_A):,}")
print(f"  Edges remaining           : {len(df_pass_A):,}")
print()
print(f"After Filter B (HPA retinal expression):")
print(f"  Edges removed             : {len(df_fail_B):,}")
print(f"  Edges remaining           : {len(df_pass_B):,}")
print()

reduction_A = (1 - len(df_pass_A)/len(df_interactions)) * 100
reduction_AB = (1 - len(df_pass_B)/len(df_interactions)) * 100

print(f"Network reduction after Filter A    : {reduction_A:.1f}%")
print(f"Network reduction after Filters A+B : {reduction_AB:.1f}%")
print()

# Proteins remaining
proteins_remaining = (
    set(df_pass_B["preferredName_A"]) |
    set(df_pass_B["preferredName_B"])
)
seeds_remaining = {
    p for p in proteins_remaining
    if p.upper() in seed_upper
}
print(f"Proteins in filtered network : {len(proteins_remaining):,}")
print(f"  RP seed proteins retained  : {len(seeds_remaining)}")
print(f"  Partners retained          : {len(proteins_remaining) - len(seeds_remaining):,}")

FILTERING PROGRESS SUMMARY

Original network (Stage 4)  : 9,835 edges

After Filter A (COMPARTMENTS):
  Edges removed (no overlap): 523
  Edges remaining           : 9,312

After Filter B (HPA retinal expression):
  Edges removed             : 389
  Edges remaining           : 8,923

Network reduction after Filter A    : 5.3%
Network reduction after Filters A+B : 9.3%

Proteins in filtered network : 3,738
  RP seed proteins retained  : 105
  Partners retained          : 3,633


In [14]:
# ─────────────────────────────────────────────────────────────────
# Notebook 05 — Cell 11: Save filtered network and update Drive
# ─────────────────────────────────────────────────────────────────

from google.colab import drive
import shutil
drive.mount("/content/drive")

# The filtered network after A + B is our working filtered network
# IID and TissueNet will add annotation scores but not hard-remove
# Save current filtered network as the primary output
df_pass_B.to_csv(
    "/content/rp_network/data/processed/network_filtered.tsv",
    sep="\t", index=False
)

# Save to Drive
shutil.copytree(
    "/content/rp_network",
    "/content/drive/MyDrive/rp_network",
    dirs_exist_ok=True
)

print("✓ Filtered network saved.")
print("✓ All files saved to Google Drive.")
print()
print("Files produced:")
print("  data/raw/compartments_human.tsv")
print("  data/raw/hpa_rna_consensus.tsv.zip")
print("  data/processed/filter_A_compartments.tsv")
print("  data/processed/filter_A_passed.tsv")
print("  data/processed/filter_B_expression.tsv")
print("  data/processed/filter_B_passed.tsv")
print("  data/processed/network_filtered.tsv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Filtered network saved.
✓ All files saved to Google Drive.

Files produced:
  data/raw/compartments_human.tsv
  data/raw/hpa_rna_consensus.tsv.zip
  data/processed/filter_A_compartments.tsv
  data/processed/filter_A_passed.tsv
  data/processed/filter_B_expression.tsv
  data/processed/filter_B_passed.tsv
  data/processed/network_filtered.tsv


In [16]:
from google.colab import files
files.download("/content/rp_network/data/processed/network_filtered.tsv")
files.download("/content/rp_network/data/processed/filter_A_compartments.tsv")
files.download("/content/rp_network/data/processed/filter_B_expression.tsv")
files.download("/content/rp_network/docs/progress_log.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>